# 23N Field Tests #07 · Ships at anchor, seen by radar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft07-ships-at-anchor-radar/ft07-ships-at-anchor-radar/ft07_ships_at_anchor_radar.ipynb)

**Question.** How many ships wait at anchor off Mundra and Kandla in the Gulf of Kutch each week, counted from radar
satellites that see through cloud and the monsoon? And how does the count change in the monsoon months?

**Why it matters.** Waiting at anchor is time a ship is paid for but not working. In 2024-25 the average
pre-berthing detention at Deendayal Port (Kandla) was 68.78 hours, against 23.38 hours across India's major
ports (Ministry of Ports, Shipping and Waterways, *Update on Indian Ports Sector up to 31.03.2025*, Table 23).
Port statistics come out months later and as averages. Radar satellites image the same water every few days,
day or night, through cloud.

**Method.** Every Sentinel-1 radar pass over the Gulf of Kutch in 2024–2025 (cross-polarised VH backscatter,
Ground Range Detected, about 20 m after resampling). Two zones: Mundra's five published anchorage areas
(corner coordinates from the port's own *Port Information Booklet*, rev. 30 Jun 2025, plus a 500 m buffer), and a
Kandla–Tuna study box that **we** drew over open water off Tuna and the Kandla creek mouth (we could not find
published anchorage coordinates for Kandla, so treat that box as an assumption). Land, jetties and fixed structures
are removed with a mask from the median of all passes. Ships are found with a classic adaptive-threshold detector
(cell-averaging CFAR: a pixel cluster much brighter than the sea around it). No model is trained and no labels are
used, so it runs on a free CPU. Counts per pass become a weekly index.

**Data and licences.** Copernicus Sentinel-1 GRD, served as Cloud-Optimised GeoTIFFs by the Microsoft Planetary
Computer STAC API: Copernicus Sentinel data terms (free, full and open, including commercial use; attribution
"Contains modified Copernicus Sentinel data 2024–2025"). **No account and no key**: the `planetary-computer` package
fetches a free, anonymous, short-lived access token. No AI model weights are used.

**Runtime.** Roughly 10–25 minutes on free Colab or Kaggle CPU, nearly all of it downloading only the two small
windows from each radar scene. `MAX_PASSES` caps it if needed. No paid API keys.

*Independent study on public data by 23N AI Labs. Mundra and Kandla are used only as locations: this says nothing
about any port operator, shipping line or government body, and implies no affiliation with any of them. All counts
are estimates from a stated detector.*

## 1 · Setup

In [ ]:
# CELL: deps
# pystac-client finds the radar scenes, planetary-computer signs the download links (anonymous, free),
# rasterio reads just the windows we need from each Cloud-Optimised GeoTIFF.
import os, sys, subprocess
if os.environ.get("FT07_SMOKE_TEST", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client", "planetary-computer", "rasterio",
                    "gradio"], check=False)

In [ ]:
# CELL: config
import os, sys, io, json, math, time, textwrap, subprocess, warnings, urllib.request, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet: it replaces the radar download with
# synthetic sea clutter and synthetic ships on the same grids. Never post its output.
SMOKE_TEST = os.environ.get("FT07_SMOKE_TEST", "0") == "1"

FT_NUM = "07"
DATA_START, DATA_END = "2024-01-01", "2025-12-31"   # two full calendar years of passes
TEST_YEAR = 2025          # headline year
ROBUST_YEAR = 2024        # robustness variant: the same numbers for the year before
MONSOON_MONTHS = [6, 7, 8, 9]   # June-September, the south-west monsoon

# ---- Radar data
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-1-grd"
POL = "vh"                # cross-polarised: ships stand out more against the sea than in VV
INSTRUMENT_MODE = "IW"    # Interferometric Wide swath, the standard mode over land and coasts

# ---- Zone 1: Mundra's published anchorage areas. Corner coordinates (degrees, decimal minutes) as printed in the
# Mundra Port Information Booklet, Section 3.6 "Port Anchorage", reviewed 30 Jun 2025. The booklet prints the VLCC
# corner D as "22039.40'N", read here as 22° 39.40'. Each area is the convex hull of its four corners.
MUNDRA_ANCHORAGES = {
    "SPM/STS":   [(22, 38.55, 69, 38.07), (22, 37.60, 69, 38.06), (22, 38.55, 69, 36.09), (22, 37.96, 69, 36.07)],
    "General A": [(22, 39.70, 69, 42.95), (22, 38.65, 69, 43.75), (22, 40.75, 69, 47.20), (22, 41.85, 69, 46.30)],
    "General B": [(22, 39.88, 69, 32.43), (22, 39.00, 69, 32.43), (22, 38.25, 69, 35.01), (22, 39.88, 69, 35.01)],
    "LNG":       [(22, 39.88, 69, 35.01), (22, 38.25, 69, 35.01), (22, 37.93, 69, 36.07), (22, 39.88, 69, 36.07)],
    "VLCC":      [(22, 38.68, 69, 41.95), (22, 37.88, 69, 42.55), (22, 38.65, 69, 43.75), (22, 39.40, 69, 43.15)],
}
MUNDRA_BUFFER_M = 500     # assumption: ships swing on their anchors and the areas' edges are not walls
# ---- Zone 2: Kandla-Tuna study box. ASSUMPTION: we found no published anchorage coordinates for Kandla, so this is
# a box we drew over open water off Tuna (22.91 N, 70.10 E) and the Kandla creek mouth. Land inside it is masked.
KANDLA_BOX = (69.95, 22.70, 70.15, 22.88)   # lon_min, lat_min, lon_max, lat_max

# ---- Grid and detector (all named assumptions, printed with the result)
GRID_CRS = "EPSG:32642"   # UTM zone 42N (66-72° E)
GRID_RES_M = 20           # metres per pixel after resampling (GRD-H IW is 10 m pixels, ~20 m resolution)
READ_DECIMATION = 2       # read every 2x2 block as one pixel (uses the COG overview): 4x less download
ZONE_MARGIN_M = 1500      # grid extends beyond each zone so the detector sees sea around ships at the edge
CFAR_GUARD_PX = 21        # 420 m guard window: bigger than the longest ships, so a ship doesn't hide itself
CFAR_OUTER_PX = 61        # 1.2 km background window
CFAR_K = 5.0              # detection: pixel > 5x the local sea background (~7 dB)
CFAR_K_ROBUST = 8.0       # robustness: a stricter ~9 dB threshold
OUTLIER_X = 4.0           # pixels brighter than 4x the scene's sea median are left out of the background
MIN_PIX, MAX_PIX = 3, 600 # a ship covers 3..600 pixels (1,200 m² .. 0.24 km²) at 20 m
STATIC_MARGIN_DB = 6.0    # median of all passes > sea + 6 dB → land, jetty or fixed structure (masked)
STATIC_DILATE_PX = 3      # mask grows by 60 m; anything touching it (e.g. ships at a jetty) is not counted
MIN_COVER = 0.90          # a pass counts for a zone only if ≥ 90 % of the zone has valid data
WORKERS = 8               # parallel downloads
MAX_PASSES = None         # e.g. 120 to cap runtime; None = every pass
DB_STEP = 0.4             # radar stack is stored as uint8 dB in 0.4 dB steps to save memory

# ---- Documented facts the post leads with (not computed here; source in the README)
PBDT_KANDLA_H, PBDT_MAJOR_H = 68.78, 23.38   # pre-berthing detention, hours, FY 2024-25, Table 23
PBDT_SOURCE = "Ministry of Ports, Shipping and Waterways, Update on Indian Ports Sector up to 31.03.2025, Table 23"

# ---- Sanity-check bounds (what is already known)
MIN_PASSES_PER_YEAR = 30           # Sentinel-1 images the Gulf every few days; fewer passes = search or data gap
MEAN_PER_PASS_RANGE = (0.5, 60)    # ships per pass in Mundra's anchorages: 0 means a broken detector, 60+ clutter
MAX_STATIC_FRAC_MUNDRA = 0.15      # the anchorages are in ~30 m of water; lots of "land" there = geolocation error
MAX_STATIC_FRAC_KANDLA = 0.60      # if the Kandla box is mostly land, the box assumption is wrong
LEN_MEDIAN_RANGE_M = (40, 400)     # median detected length; cargo ships are ~100-300 m
MAX_MIN_SIZE_SHARE = 0.5           # if > 50 % of detections are the minimum size, it is counting speckle
PLATFORM_RATIO_RANGE = (0.67, 1.5) # Sentinel-1A vs 1C mean counts over the same months (sensor/version break)
ORBIT_RATIO_RANGE = (0.5, 2.0)     # ascending vs descending passes (look direction, time of day)
MAX_MONSOON_CHANGE = 0.80          # a monsoon change beyond ±80 % is more likely sea clutter than ships
SEA_STEP_DB = 3.0                  # month-to-month jump in sea background → calibration/processing change (note)

# ---- Where things live (this branch until it is merged)
REPO_REF = "ft07-ships-at-anchor-radar"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"

# ---- Brand kit
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
DATA_CREDIT = "Contains modified Copernicus Sentinel data 2024-2025, via Microsoft Planetary Computer"
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic radar scenes. These results are NOT real. Do NOT post.")

In [ ]:
# CELL: zones
# Builds a 20 m UTM grid around each zone and a mask of the water we count ships in.
import rasterio
from rasterio.features import rasterize
from rasterio.warp import transform as warp_transform
from affine import Affine

def dm(deg, minutes):
    return deg + minutes / 60.0

def convex_hull(pts):
    pts = sorted(set(pts))
    if len(pts) <= 2:
        return pts
    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    lower, upper = [], []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)
    return lower[:-1] + upper[:-1]

def to_utm(lons, lats):
    x, y = warp_transform("EPSG:4326", GRID_CRS, list(lons), list(lats))
    return np.array(x), np.array(y)

def to_lonlat(xs, ys):
    lon, lat = warp_transform(GRID_CRS, "EPSG:4326", list(xs), list(ys))
    return np.array(lon), np.array(lat)

def make_zone(name, polys_ll, buffer_m=0):
    # polys_ll: list of [(lon, lat), ...] rings
    allx, ally, rings = [], [], []
    for ring in polys_ll:
        x, y = to_utm([p[0] for p in ring], [p[1] for p in ring])
        rings.append(list(zip(x, y)) + [(x[0], y[0])])
        allx += list(x); ally += list(y)
    m = ZONE_MARGIN_M + buffer_m
    x0 = math.floor((min(allx) - m) / GRID_RES_M) * GRID_RES_M
    y1 = math.ceil((max(ally) + m) / GRID_RES_M) * GRID_RES_M
    W = int(math.ceil((max(allx) + m - x0) / GRID_RES_M)); H = int(math.ceil((y1 - (min(ally) - m)) / GRID_RES_M))
    tr = Affine(GRID_RES_M, 0, x0, 0, -GRID_RES_M, y1)
    mask = rasterize([({"type": "Polygon", "coordinates": [r]}, 1) for r in rings], out_shape=(H, W),
                     transform=tr, fill=0, dtype="uint8").astype(bool)
    if buffer_m:
        r = int(round(buffer_m / GRID_RES_M)); yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
        mask = ndimage.binary_dilation(mask, structure=(xx ** 2 + yy ** 2) <= r * r)
    # outline points of the grid (edges sampled every ~1 km) in lon/lat, used to find the window in each scene
    ex = np.r_[np.linspace(0, W, 30), np.full(30, W), np.linspace(W, 0, 30), np.zeros(30)]
    ey = np.r_[np.zeros(30), np.linspace(0, H, 30), np.full(30, H), np.linspace(H, 0, 30)]
    gx, gy = tr * (ex, ey)
    lon, lat = to_lonlat(gx, gy)
    return dict(name=name, H=H, W=W, transform=tr, mask=mask, rings=rings, edge_lon=lon, edge_lat=lat,
                area_km2=mask.sum() * GRID_RES_M ** 2 / 1e6)

mundra_polys = [convex_hull([(dm(c[2], c[3]), dm(c[0], c[1])) for c in corners]) for corners in MUNDRA_ANCHORAGES.values()]
lo0, la0, lo1, la1 = KANDLA_BOX
ZONES = {
    "Mundra": make_zone("Mundra anchorages", mundra_polys, MUNDRA_BUFFER_M),
    "Kandla": make_zone("Kandla-Tuna box", [[(lo0, la0), (lo1, la0), (lo1, la1), (lo0, la1)]]),
}
ZONE_ORDER = ["Mundra", "Kandla"]
AOI_BBOX = [min(z["edge_lon"].min() for z in ZONES.values()), min(z["edge_lat"].min() for z in ZONES.values()),
            max(z["edge_lon"].max() for z in ZONES.values()), max(z["edge_lat"].max() for z in ZONES.values())]
for k, z in ZONES.items():
    print(f"{z['name']}: grid {z['W']}x{z['H']} px at {GRID_RES_M} m, counting area {z['area_km2']:.1f} km²")
print("Search bbox (lon/lat):", np.round(AOI_BBOX, 3))

## 2 · Data: every Sentinel-1 pass over the Gulf of Kutch, 2024–2025

The Planetary Computer's STAC API lists every Sentinel-1 GRD scene. Each scene's VH band is a Cloud-Optimised
GeoTIFF of roughly 1 GB, but we read only the two small windows around our zones (at 20 m, via the file's built-in
overview), so each pass costs a few megabytes. GRD images are in radar geometry with ground control points
(GCPs) rather than a map projection; the reader uses the GCPs near each window to resample it onto our UTM grid.
Slices of the same pass are mosaicked. Values are uncalibrated digital numbers squared (intensity): the detector
only ever uses **ratios** within one image, so calibration differences between Sentinel-1A, 1C and 1D cannot shift
the counts directly.

In [ ]:
# CELL: loaders
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from rasterio.transform import GCPTransformer, rowcol as affine_rowcol
from rasterio.control import GroundControlPoint

def to_u8(inten):
    with np.errstate(divide="ignore", invalid="ignore"):
        db = 10 * np.log10(inten)
    q = np.clip(np.round(db / DB_STEP) + 1, 1, 255)
    return np.where(np.isfinite(db), q, 0).astype(np.uint8)

def from_u8(q):
    return np.where(q > 0, 10 ** (((q.astype(np.float32) - 1) * DB_STEP) / 10), np.nan).astype(np.float32)

def u8_db(q):
    return np.where(q > 0, (q.astype(np.float32) - 1) * DB_STEP, np.nan)

def read_zone_from_src(src, zone, decim=READ_DECIMATION, pad=64, gcp_margin=3000):
    # Returns the zone grid (float32 intensity, NaN = no data) or None if the scene misses the zone.
    gcps, gcrs = src.gcps
    use_gcps = bool(gcps) and (src.crs is None or src.transform.is_identity)
    if use_gcps:
        rows, cols = GCPTransformer(gcps).rowcol(zone["edge_lon"], zone["edge_lat"])
    else:
        x, y = warp_transform("EPSG:4326", src.crs, list(zone["edge_lon"]), list(zone["edge_lat"]))
        rows, cols = affine_rowcol(src.transform, x, y)
    rows, cols = np.asarray(rows), np.asarray(cols)
    r0, r1 = max(0, int(rows.min()) - pad), min(src.height, int(rows.max()) + pad)
    c0, c1 = max(0, int(cols.min()) - pad), min(src.width, int(cols.max()) + pad)
    h, w = (r1 - r0) // decim, (c1 - c0) // decim
    if h < 16 or w < 16:
        return None
    win = Window(c0, r0, w * decim, h * decim)
    dn = src.read(1, window=win, out_shape=(h, w), resampling=Resampling.average).astype(np.float32)
    inten = np.where(dn > 0, dn * dn, np.nan).astype(np.float32)
    if not np.isfinite(inten).any():
        return None
    dst = np.full((zone["H"], zone["W"]), np.nan, np.float32)
    if use_gcps:
        sel = [g for g in gcps if r0 - gcp_margin <= g.row <= r1 + gcp_margin and c0 - gcp_margin <= g.col <= c1 + gcp_margin]
        if len(sel) < 6:
            sel = list(gcps)
        loc = [GroundControlPoint(row=(g.row - r0) / decim, col=(g.col - c0) / decim, x=g.x, y=g.y, z=g.z) for g in sel]
        reproject(inten, dst, gcps=loc, src_crs=gcrs, src_nodata=np.nan, dst_transform=zone["transform"],
                  dst_crs=GRID_CRS, dst_nodata=np.nan, resampling=Resampling.bilinear)
    else:
        wtr = src.window_transform(win) * Affine.scale(decim)
        reproject(inten, dst, src_transform=wtr, src_crs=src.crs, src_nodata=np.nan, dst_transform=zone["transform"],
                  dst_crs=GRID_CRS, dst_nodata=np.nan, resampling=Resampling.bilinear)
    return dst

GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="2",
                GDAL_HTTP_MULTIRANGE="YES", GDAL_HTTP_MERGE_CONSECUTIVE_RANGES="YES", VSI_CACHE="TRUE")

def read_zone(href, zone, tries=3):
    import planetary_computer as pc
    last = None
    for k in range(tries):
        try:
            with rasterio.Env(**GDAL_ENV):
                with rasterio.open(pc.sign(href)) as src:      # signed just before use: tokens expire
                    return read_zone_from_src(src, zone)
        except Exception as e:
            last = e; time.sleep(2 * (k + 1))
    raise last

def read_pass(hrefs):
    out, errs = {}, []
    for zk in ZONE_ORDER:
        z = ZONES[zk]; acc = None
        for href in hrefs:
            try:
                g = read_zone(href, z)
            except Exception as e:
                errs.append(f"{zk}: {type(e).__name__}: {str(e)[:120]}"); continue
            if g is not None:
                acc = g if acc is None else np.where(np.isfinite(acc), acc, g)
        if acc is not None and np.isfinite(acc[z["mask"]]).mean() >= MIN_COVER:
            out[zk] = to_u8(acc)
    return out, errs

def utc_naive(t):
    t = pd.Timestamp(t)
    return t.tz_convert("UTC").tz_localize(None) if t.tzinfo else t

def search_passes():
    import pystac_client
    cat = pystac_client.Client.open(STAC_URL)
    s = cat.search(collections=[COLLECTION], bbox=AOI_BBOX, datetime=f"{DATA_START}/{DATA_END}",
                   query={"sar:instrument_mode": {"eq": INSTRUMENT_MODE}})
    rows = []
    for it in s.items():
        if POL not in it.assets:
            continue
        p = it.properties
        rows.append(dict(id=it.id, time=utc_naive(it.datetime), platform=str(p.get("platform", "?")).upper(),
                         orbit=str(p.get("sat:orbit_state", "?")).lower(), rel_orbit=p.get("sat:relative_orbit"),
                         href=it.assets[POL].href))
    items = pd.DataFrame(rows)
    if items.empty:
        return items, items
    items["pass_key"] = (items["platform"] + "|" + items["orbit"] + "|" + items["rel_orbit"].astype(str) + "|" +
                         items["time"].dt.strftime("%Y-%m-%d"))
    passes = (items.sort_values("time").groupby("pass_key")
              .agg(time=("time", "min"), platform=("platform", "first"), orbit=("orbit", "first"),
                   rel_orbit=("rel_orbit", "first"), hrefs=("href", list), n_slices=("id", "size"))
              .sort_values("time").reset_index())
    return items, passes

def load_real():
    items, passes = search_passes()
    print(f"STAC: {len(items)} scenes with {POL.upper()} → {len(passes)} passes")
    if MAX_PASSES and len(passes) > MAX_PASSES:
        passes = passes.iloc[np.linspace(0, len(passes) - 1, MAX_PASSES).round().astype(int)].reset_index(drop=True)
        print(f"MAX_PASSES: using {len(passes)} evenly spaced passes")
    stacks = {zk: np.zeros((len(passes), ZONES[zk]["H"], ZONES[zk]["W"]), np.uint8) for zk in ZONE_ORDER}
    cov = {zk: np.zeros(len(passes), bool) for zk in ZONE_ORDER}
    errors, t0 = [], time.time()
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(read_pass, h): i for i, h in enumerate(passes["hrefs"])}
        for n, f in enumerate(as_completed(futs), 1):
            i = futs[f]; out, errs = f.result(); errors += errs
            for zk, q in out.items():
                stacks[zk][i] = q; cov[zk][i] = True
            if n % 25 == 0 or n == len(passes):
                print(f"  {n}/{len(passes)} passes read · {time.time() - t0:.0f}s · read errors so far: {len(errors)}")
    for zk in ZONE_ORDER:
        passes[f"cov_{zk}"] = cov[zk]
    return passes.drop(columns=["hrefs"]), stacks, errors

def synthetic_data(seed=7):
    # SMOKE TEST ONLY: speckled sea (rougher in the monsoon), a strip of land, three fixed bright structures,
    # and a Poisson number of bright ship-shaped blobs per pass. Not real data.
    rng = np.random.default_rng(seed)
    rows = []
    for d in pd.date_range(DATA_START, DATA_END, freq="D"):
        k = d.dayofyear
        if k % 12 == 0: rows.append((d + pd.Timedelta(hours=1, minutes=10), "SENTINEL-1A", "descending", 34))
        if k % 12 == 6: rows.append((d + pd.Timedelta(hours=13, minutes=20), "SENTINEL-1A", "ascending", 100))
        if d >= pd.Timestamp("2025-04-01") and k % 12 == 3:
            rows.append((d + pd.Timedelta(hours=1, minutes=10), "SENTINEL-1C", "descending", 34))
    passes = pd.DataFrame(rows, columns=["time", "platform", "orbit", "rel_orbit"])
    passes["pass_key"] = [f"synthetic-{i}" for i in range(len(passes))]; passes["n_slices"] = 1
    base = {"Mundra": 14.0, "Kandla": 9.0}
    stacks = {zk: np.zeros((len(passes), ZONES[zk]["H"], ZONES[zk]["W"]), np.uint8) for zk in ZONE_ORDER}
    cov = {zk: np.zeros(len(passes), bool) for zk in ZONE_ORDER}
    for zk in ZONE_ORDER:
        z = ZONES[zk]; H, W = z["H"], z["W"]
        land = np.zeros((H, W), bool)
        if zk == "Kandla":
            land[: int(0.12 * H)] = True
        fixed = [(int(H * f1), int(W * f2)) for f1, f2 in ((0.5, 0.2), (0.4, 0.55), (0.6, 0.8))]
        cand = np.argwhere(z["mask"] & ~land)
        for i, t in enumerate(passes["time"]):
            if rng.random() < 0.08:
                continue                                    # zone not covered by this pass
            mon = t.month in MONSOON_MONTHS
            sea = rng.gamma(10, 0.1, (H, W)).astype(np.float32) * 1500 * (1.8 if mon else 1.0)
            img = np.where(land, rng.gamma(4, 0.25, (H, W)) * 60000, sea)
            for (r, c) in fixed:
                img[r - 2:r + 3, c - 2:c + 3] = 1500 * 200
            lam = base[zk] * (1 + 0.2 * np.cos(2 * np.pi * (t.dayofyear - 30) / 365)) * (0.8 if mon else 1.0)
            for _ in range(rng.poisson(lam)):
                r, c = cand[rng.integers(len(cand))]
                L, ang, amp = rng.uniform(5, 13), rng.uniform(0, np.pi), rng.uniform(15, 60) * 1500
                for s in np.linspace(-L / 2, L / 2, int(L * 3)):
                    rr, cc = int(round(r + s * np.sin(ang))), int(round(c + s * np.cos(ang)))
                    img[max(rr - 1, 0):rr + 1, max(cc - 1, 0):cc + 1] = amp
            stacks[zk][i] = to_u8(img); cov[zk][i] = True
    for zk in ZONE_ORDER:
        passes[f"cov_{zk}"] = cov[zk]
    return passes, stacks, []

In [ ]:
# CELL: load_data
CACHE = f"ft{FT_NUM}_cache.npz"
if SMOKE_TEST:
    PASSES, STACK, READ_ERRORS = synthetic_data()
elif os.path.exists(CACHE):
    _c = np.load(CACHE, allow_pickle=False)
    STACK = {zk: _c[zk] for zk in ZONE_ORDER}
    PASSES = pd.read_csv(CACHE.replace(".npz", "_passes.csv"), parse_dates=["time"]); READ_ERRORS = []
    print("Loaded the radar stack from the local cache (delete", CACHE, "to download again).")
else:
    PASSES, STACK, READ_ERRORS = load_real()
    np.savez_compressed(CACHE, **STACK); PASSES.to_csv(CACHE.replace(".npz", "_passes.csv"), index=False)

PASSES["year"] = PASSES["time"].dt.year
PASSES["monsoon"] = PASSES["time"].dt.month.isin(MONSOON_MONTHS)
print(f"{len(PASSES)} passes, {PASSES['time'].min():%d %b %Y} → {PASSES['time'].max():%d %b %Y}")
print(PASSES.groupby(["year", "platform"]).size().unstack(fill_value=0))
for zk in ZONE_ORDER:
    print(f"{zk}: {int(PASSES[f'cov_{zk}'].sum())} passes cover ≥ {MIN_COVER:.0%} of the zone")
if READ_ERRORS:
    print(f"{len(READ_ERRORS)} read errors (first 3):", READ_ERRORS[:3])
MONTHLY_PASSES = PASSES.set_index("time").resample("MS").size()
EMPTY_MONTHS = [f"{t:%b %Y}" for t, n in MONTHLY_PASSES.items() if n == 0]
print("Months with no pass:", EMPTY_MONTHS or "none")

## 3 · Method: mask the land, then find bright clusters on dark sea

1. **Static mask.** For each zone we take the median of every pass. Land, jetties, buoys and platforms are bright in
   most passes; a ship is rarely in the same spot for half of them. Pixels whose median is `STATIC_MARGIN_DB` above
   the typical sea are masked, grown by `STATIC_DILATE_PX`, and anything touching the mask (a ship alongside a
   jetty, for example) is not counted.
2. **Detector (CA-CFAR).** For every pixel, the local sea background is the mean intensity in a 1.2 km window,
   leaving out a 420 m guard window around the pixel and any already-bright pixels. A pixel is a candidate if it is
   more than `CFAR_K` times its background. Touching candidates form one object; objects of `MIN_PIX`–`MAX_PIX`
   pixels are ships. Their length is estimated from the object's extent.
3. **Counts.** A ship is counted in a zone if its centre falls in the zone's counting area (Mundra: the five
   published anchorages plus 500 m; Kandla: the box, minus land). A pass contributes only if it covers at least
   `MIN_COVER` of the zone. The **weekly index** is the mean count per pass in each week, summed over the two zones.

A single radar image can't tell a ship at anchor from one moving slowly through the zone; in the anchorages most
are at anchor, but the count is "ships present", not "ships waiting". This is said again in the limitations.

In [ ]:
# CELL: detect
def nanmedian_stack(st, sel, chunk=48):
    H = st.shape[1]; out = np.full(st.shape[1:], np.nan, np.float32)
    idx = np.where(sel)[0]
    for r in range(0, H, chunk):
        block = u8_db(st[idx, r:r + chunk])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            out[r:r + chunk] = np.nanmedian(block, axis=0)
    return out

MASKS = {}
for zk in ZONE_ORDER:
    z = ZONES[zk]; sel = PASSES[f"cov_{zk}"].values
    med = nanmedian_stack(STACK[zk], sel)
    finite = np.isfinite(med)
    sea_db = float(np.nanpercentile(med[finite], 30)) if finite.any() else np.nan
    static = ~finite | (med > sea_db + STATIC_MARGIN_DB)
    static = ndimage.binary_dilation(static, iterations=STATIC_DILATE_PX)
    border = ndimage.binary_dilation(static, iterations=2) & ~static
    count_area = z["mask"] & ~static
    MASKS[zk] = dict(median_db=med, sea_db=sea_db, static=static, border=border, count_area=count_area,
                     static_frac_zone=float(static[z["mask"]].mean()),
                     count_km2=count_area.sum() * GRID_RES_M ** 2 / 1e6)
    print(f"{zk}: typical sea {sea_db:.1f} dB (uncalibrated), static mask covers "
          f"{MASKS[zk]['static_frac_zone']:.1%} of the zone, counting area {MASKS[zk]['count_km2']:.1f} km²")

def cfar_detect(inten, m, k):
    water = np.isfinite(inten) & ~m["static"]
    if water.sum() < 1000:
        return [], np.nan
    sea_med = float(np.median(inten[water]))
    w = water & (inten < OUTLIER_X * sea_med)
    iw = np.where(w, inten, 0).astype(np.float64); wf = w.astype(np.float64)
    def box_sum(a, n):
        return ndimage.uniform_filter(a, n, mode="constant") * (n * n)
    num = box_sum(iw, CFAR_OUTER_PX) - box_sum(iw, CFAR_GUARD_PX)
    den = box_sum(wf, CFAR_OUTER_PX) - box_sum(wf, CFAR_GUARD_PX)
    bg = np.where(den > 50, num / np.maximum(den, 1), sea_med)
    hit = water & (inten > k * bg)
    lab, n = ndimage.label(hit, structure=np.ones((3, 3)))
    dets = []
    if n == 0:
        return dets, 10 * np.log10(sea_med)
    sizes = np.bincount(lab.ravel(), minlength=n + 1)
    touches = np.unique(lab[m["border"] & hit])
    for i, sl in enumerate(ndimage.find_objects(lab), start=1):
        if sl is None or not (MIN_PIX <= sizes[i] <= MAX_PIX) or i in touches:
            continue
        rr, cc = np.nonzero(lab[sl] == i)
        r, c = rr.mean() + sl[0].start, cc.mean() + sl[1].start
        if not m["count_area"][int(round(r)), int(round(c))]:
            continue
        ext = math.hypot(np.ptp(rr) + 1, np.ptp(cc) + 1) * GRID_RES_M
        peak = float(inten[sl][lab[sl] == i].max() / bg[int(round(r)), int(round(c))])
        dets.append(dict(row=r, col=c, npix=int(sizes[i]), length_m=ext, peak_ratio=peak))
    return dets, 10 * np.log10(sea_med)

rows, sea_rows, t0 = [], [], time.time()
for zk in ZONE_ORDER:
    z, m = ZONES[zk], MASKS[zk]
    for i in np.where(PASSES[f"cov_{zk}"].values)[0]:
        inten = from_u8(STACK[zk][i])
        for k in (CFAR_K, CFAR_K_ROBUST):
            dets, sea = cfar_detect(inten, m, k)
            for d in dets:
                rows.append(dict(pass_idx=i, zone=zk, k=k, **d))
            if k == CFAR_K:
                sea_rows.append(dict(pass_idx=i, zone=zk, sea_db=sea))
DET = pd.DataFrame(rows, columns=["pass_idx", "zone", "k", "row", "col", "npix", "length_m", "peak_ratio"])
SEA = pd.DataFrame(sea_rows)
for zk in ZONE_ORDER:                                   # positions in lon/lat for maps and the demo
    s = DET.zone == zk
    if s.any():
        x, y = ZONES[zk]["transform"] * (DET.loc[s, "col"].values + 0.5, DET.loc[s, "row"].values + 0.5)
        DET.loc[s, "lon"], DET.loc[s, "lat"] = to_lonlat(x, y)

def counts_for(k):
    c = DET[DET.k == k].groupby(["pass_idx", "zone"]).size().unstack(fill_value=0).reindex(columns=ZONE_ORDER, fill_value=0)
    c = c.reindex(range(len(PASSES)), fill_value=0).astype(float)
    for zk in ZONE_ORDER:
        c.loc[~PASSES[f"cov_{zk}"].values, zk] = np.nan   # not observed ≠ zero ships
    return c

COUNTS, COUNTS_ROBUST = counts_for(CFAR_K), counts_for(CFAR_K_ROBUST)
print(f"Detector ran on {len(SEA)} zone-passes in {time.time() - t0:.0f}s: {int((DET.k == CFAR_K).sum())} detections at K={CFAR_K}")
print(COUNTS.describe().round(1))

## 4 · Results: the headline, after the sanity checks

In [ ]:
# CELL: results
P = PASSES.join(COUNTS)
PR = PASSES.join(COUNTS_ROBUST)

def zone_means(df, mask=None):
    d = df if mask is None else df[mask]
    return {zk: float(d[zk].mean()) for zk in ZONE_ORDER}

def combined(df, mask=None):
    zm = zone_means(df, mask)
    return sum(zm.values()), zm

def monsoon_change(df, year):
    y = df.year == year
    mon, _ = combined(df, y & df.monsoon); rest, _ = combined(df, y & ~df.monsoon)
    return mon, rest, (mon / rest - 1) if rest > 0 else float("nan")

MEAN_T, ZM_T = combined(P, P.year == TEST_YEAR)
MEAN_R, ZM_R = combined(P, P.year == ROBUST_YEAR)
MON_T, REST_T, CHG_T = monsoon_change(P, TEST_YEAR)
MON_R, REST_R, CHG_R = monsoon_change(P, ROBUST_YEAR)
_, _, CHG_T_STRICT = monsoon_change(PR, TEST_YEAR)
MEAN_T_STRICT, _ = combined(PR, PR.year == TEST_YEAR)

# weekly index: mean count per pass in each week, per zone, summed only for weeks where both zones were seen
WEEKLY_Z = P.set_index("time")[ZONE_ORDER].resample("W-MON", label="left", closed="left").mean()
WEEKLY = WEEKLY_Z.sum(axis=1, min_count=len(ZONE_ORDER))
WEEKLY_T = WEEKLY[WEEKLY.index.year == TEST_YEAR].dropna()
PEAK_WEEK, PEAK_VAL = (WEEKLY_T.idxmax(), float(WEEKLY_T.max())) if len(WEEKLY_T) else (pd.NaT, float("nan"))
N_WEEKS_SEEN = int(WEEKLY_T.notna().sum())

# splits used by the checks
def ratio_split(col, a, b, months=None):
    d = P[P.year == TEST_YEAR]
    if months is not None:
        d = d[d.time.dt.month.isin(months)]
    ma, _ = combined(d, d[col] == a); mb, _ = combined(d, d[col] == b)
    return ma, mb, (ma / mb if mb > 0 else float("nan"))
PLAT_MONTHS = sorted(P[(P.year == TEST_YEAR) & P.platform.str.contains("1C")].time.dt.month.unique())
PLAT_A, PLAT_C, PLAT_RATIO = ratio_split("platform", "SENTINEL-1A", "SENTINEL-1C", PLAT_MONTHS) if PLAT_MONTHS else (np.nan, np.nan, np.nan)
ORB_A, ORB_D, ORB_RATIO = ratio_split("orbit", "ascending", "descending")

D0 = DET[DET.k == CFAR_K]
LEN_MED = float(D0.length_m.median()) if len(D0) else float("nan")
MIN_SHARE = float((D0.npix <= MIN_PIX).mean()) if len(D0) else float("nan")
SEA_M = (SEA[SEA.zone == "Mundra"].assign(time=lambda s: PASSES.time.values[s.pass_idx.values])
         .set_index("time")["sea_db"].resample("MS").median().dropna())
SEA_JUMPS = [(t, float(v)) for t, v in SEA_M.diff().abs().items() if v > SEA_STEP_DB]

# ---------------- sanity checks (before any headline is printed) ----------------
PROBLEMS, NOTES = [], []
for y in (ROBUST_YEAR, TEST_YEAR):
    for zk in ZONE_ORDER:
        n = int(P[(P.year == y) & P[f"cov_{zk}"]].shape[0])
        if n < MIN_PASSES_PER_YEAR:
            PROBLEMS.append(f"only {n} usable passes over {zk} in {y} (< {MIN_PASSES_PER_YEAR}): search, coverage or download problem")
if len(EMPTY_MONTHS) >= 2:
    PROBLEMS.append(f"{len(EMPTY_MONTHS)} months with no pass at all ({', '.join(EMPTY_MONTHS[:6])}): data gap, weekly index unreliable")
mm = ZM_T.get("Mundra", float("nan"))
if not (MEAN_PER_PASS_RANGE[0] <= mm <= MEAN_PER_PASS_RANGE[1]):
    PROBLEMS.append(f"Mundra anchorages average {mm:.1f} ships per pass, outside {MEAN_PER_PASS_RANGE}: detector broken or counting clutter")
if MASKS["Mundra"]["static_frac_zone"] > MAX_STATIC_FRAC_MUNDRA:
    PROBLEMS.append(f"{MASKS['Mundra']['static_frac_zone']:.0%} of Mundra's anchorages masked as land/structure: geolocation wrong?")
if MASKS["Kandla"]["static_frac_zone"] > MAX_STATIC_FRAC_KANDLA:
    PROBLEMS.append(f"{MASKS['Kandla']['static_frac_zone']:.0%} of the Kandla box is masked: the box is mostly land, redraw it")
if not (LEN_MEDIAN_RANGE_M[0] <= LEN_MED <= LEN_MEDIAN_RANGE_M[1]):
    PROBLEMS.append(f"median detected length {LEN_MED:.0f} m outside {LEN_MEDIAN_RANGE_M}: not ships?")
if MIN_SHARE > MAX_MIN_SIZE_SHARE:
    PROBLEMS.append(f"{MIN_SHARE:.0%} of detections are the minimum size: likely speckle, raise CFAR_K")
if PLAT_RATIO == PLAT_RATIO and not (PLATFORM_RATIO_RANGE[0] <= PLAT_RATIO <= PLATFORM_RATIO_RANGE[1]):
    PROBLEMS.append(f"Sentinel-1A vs 1C counts differ by a factor {PLAT_RATIO:.2f} over the same months: sensor/version break")
elif PLAT_RATIO != PLAT_RATIO:
    NOTES.append(f"no Sentinel-1C passes in {TEST_YEAR} in this catalogue: platform check skipped")
if ORB_RATIO == ORB_RATIO and not (ORBIT_RATIO_RANGE[0] <= ORB_RATIO <= ORBIT_RATIO_RANGE[1]):
    PROBLEMS.append(f"ascending vs descending counts differ by a factor {ORB_RATIO:.2f}: look direction or time-of-day bias")
if CHG_T == CHG_T and abs(CHG_T) > MAX_MONSOON_CHANGE:
    PROBLEMS.append(f"monsoon change {CHG_T:+.0%} is beyond ±{MAX_MONSOON_CHANGE:.0%}: more likely sea clutter than ships")
if CHG_T == CHG_T and CHG_T_STRICT == CHG_T_STRICT and np.sign(CHG_T) != np.sign(CHG_T_STRICT):
    PROBLEMS.append(f"monsoon change flips sign with the stricter threshold ({CHG_T:+.0%} vs {CHG_T_STRICT:+.0%}): the pattern is the detector, not the ships")
if CHG_T == CHG_T and CHG_R == CHG_R and np.sign(CHG_T) != np.sign(CHG_R):
    NOTES.append(f"robustness: the monsoon change has opposite signs in {TEST_YEAR} ({CHG_T:+.0%}) and {ROBUST_YEAR} ({CHG_R:+.0%}): say it is not a stable pattern")
for t, v in SEA_JUMPS:
    NOTES.append(f"sea background jumps {v:.1f} dB into {t:%b %Y} (processing/calibration change?): counts use ratios, but check that month")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE TEST: synthetic radar scenes")

print("Assumptions:", dict(POL=POL.upper(), GRID_RES_M=GRID_RES_M, CFAR_K=CFAR_K, CFAR_GUARD_PX=CFAR_GUARD_PX,
      CFAR_OUTER_PX=CFAR_OUTER_PX, MIN_PIX=MIN_PIX, MAX_PIX=MAX_PIX, STATIC_MARGIN_DB=STATIC_MARGIN_DB,
      MUNDRA_BUFFER_M=MUNDRA_BUFFER_M, KANDLA_BOX=KANDLA_BOX, MIN_COVER=MIN_COVER, MONSOON_MONTHS=MONSOON_MONTHS))
for n in NOTES:
    print("note:", n)
if PROBLEMS:
    print("\n" + "!" * 78)
    for p in PROBLEMS:
        print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every failure above is explained or fixed.")
    print("!" * 78 + "\n")
else:
    print("All sanity checks passed.")

def fmt_n(v):
    return f"{v:.1f}" if v < 10 else f"{v:.0f}"

MORE_LESS = "more" if CHG_T >= 0 else "fewer"
HEADLINE = (f"Sentinel-1 radar counted {fmt_n(MEAN_T)} ships on an average pass in the anchorage waters off Mundra "
            f"and Kandla in {TEST_YEAR} (estimate); in the monsoon months (Jun–Sep) there were {abs(CHG_T):.0%} "
            f"{MORE_LESS} than in the rest of the year.")
BIG = f"{fmt_n(MEAN_T)} ships"
BIG_LINE = f"at anchor off Mundra and Kandla on an average radar pass in {TEST_YEAR}. Monsoon: {CHG_T:+.0%}."
print("\nHEADLINE (estimate):", HEADLINE)
print(f"  by zone, {TEST_YEAR}: " + ", ".join(f"{zk} {v:.1f}/pass" for zk, v in ZM_T.items()))
print(f"  monsoon {fmt_n(MON_T)} vs rest of year {fmt_n(REST_T)} ships per pass")
if len(WEEKLY_T):
    print(f"  weekly index: {N_WEEKS_SEEN} weeks observed in {TEST_YEAR}; busiest week of {PEAK_WEEK:%d %b %Y}: {PEAK_VAL:.1f} ships per pass")
print(f"  detected length: median {LEN_MED:.0f} m (rough, from the object's extent at {GRID_RES_M} m pixels)")
print("\nRobustness:")
print(f"  {ROBUST_YEAR}: {fmt_n(MEAN_R)} ships per pass, monsoon change {CHG_R:+.0%}")
print(f"  stricter threshold K={CFAR_K_ROBUST}: {fmt_n(MEAN_T_STRICT)} ships per pass, monsoon change {CHG_T_STRICT:+.0%}")
print(f"  ascending {fmt_n(ORB_A)} vs descending {fmt_n(ORB_D)} ships per pass (ratio {ORB_RATIO:.2f})")
if PLAT_RATIO == PLAT_RATIO:
    print(f"  Sentinel-1A {fmt_n(PLAT_A)} vs 1C {fmt_n(PLAT_C)} ships per pass over the same months (ratio {PLAT_RATIO:.2f})")

In [ ]:
# CELL: results_chart
MON_SPANS = [(pd.Timestamp(y, MONSOON_MONTHS[0], 1), pd.Timestamp(y, MONSOON_MONTHS[-1], 1) + pd.offsets.MonthEnd(1))
             for y in (ROBUST_YEAR, TEST_YEAR)]

def draw_weekly(ax, upto=None, big=False):
    w = WEEKLY.dropna()
    if upto is not None:
        w = w[w.index <= upto]
    for a, b in MON_SPANS:
        ax.axvspan(a, b, color=B.PAPER_DEEP, lw=0, zorder=0)
    ax.plot(w.index, w.values, color=B.MOSS, lw=3 if big else 2, marker="o", ms=4 if big else 2.5)
    ax.set_xlim(pd.Timestamp(DATA_START), pd.Timestamp(DATA_END))
    top = np.nanmax(WEEKLY.values) if WEEKLY.notna().any() else 1
    ax.set_ylim(0, top * 1.25)
    ax.set_ylabel("ships per pass", fontsize=14 if big else 11)
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(matplotlib.dates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.tick_params(labelsize=13 if big else 10)
    ax.text(MON_SPANS[0][0] + pd.Timedelta(days=4), top * 1.17, "MONSOON", fontproperties=B.FP.get("mono"),
            fontsize=11 if big else 8, color=B.STONE, va="top")

def zone_db(zk, i):
    return u8_db(STACK[zk][i])

def draw_zone(ax, zk, i, k=CFAR_K, show_dets=True, n_show=None, outline=True):
    m = MASKS[zk]; img = zone_db(zk, i)
    ax.imshow(img, cmap=B.SEQ_CMAP, vmin=m["sea_db"] - 1, vmax=m["sea_db"] + 14, interpolation="nearest")
    if outline:
        tr = ~ZONES[zk]["transform"]
        for ring in ZONES[zk]["rings"]:
            cs, rs = zip(*[tr * p for p in ring])
            ax.plot(cs, rs, color=B.STONE, lw=1, ls="--")
    d = DET[(DET.k == k) & (DET.zone == zk) & (DET.pass_idx == i)]
    if show_dets and len(d):
        d = d if n_show is None else d.iloc[:n_show]
        ax.scatter(d.col, d.row, s=260, facecolors="none", edgecolors=B.MOSS, linewidths=2.2)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    return len(DET[(DET.k == k) & (DET.zone == zk) & (DET.pass_idx == i)])

fig, ax = plt.subplots(figsize=(11, 4.6))
draw_weekly(ax)
B.chart_title(ax, "Ships at anchor off Mundra and Kandla, by week",
              f"mean ships per Sentinel-1 pass · both zones · shaded: Jun–Sep monsoon")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT} · CA-CFAR K={CFAR_K}" + (" · SMOKE TEST, NOT REAL" if SMOKE_TEST else ""))
fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_weekly.png", dpi=150); plt.show()

# where the ships sit: detections in TEST_YEAR on Mundra's grid
zk = "Mundra"; z = ZONES[zk]
dd = DET[(DET.k == CFAR_K) & (DET.zone == zk) & (PASSES.year.values[DET.pass_idx.values] == TEST_YEAR)]
dens = np.histogram2d(dd.row, dd.col, bins=[max(1, z["H"] // 10), max(1, z["W"] // 10)],
                      range=[[0, z["H"]], [0, z["W"]]])[0]
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.imshow(np.where(MASKS[zk]["static"][::10, ::10][:dens.shape[0], :dens.shape[1]], np.nan, dens), cmap=B.SEQ_CMAP,
          extent=[0, z["W"], z["H"], 0], interpolation="nearest")
tr = ~z["transform"]
for ring in z["rings"]:
    cs, rs = zip(*[tr * p for p in ring]); ax.plot(cs, rs, color=B.AMBER, lw=1.4)
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
B.chart_title(ax, "Where the ships sit: Mundra's anchorages", f"detections per 200 m cell, {TEST_YEAR} · amber: published anchorage areas")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}" + (" · SMOKE TEST, NOT REAL" if SMOKE_TEST else ""))
fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_mundra_density.png", dpi=150); plt.show()

## 5 · Interactive demo
Pick a radar pass: you see both zones as the satellite saw them, with each detected ship circled and the count.
The cell doesn't block "Run all"; open the public `gradio.live` link it prints.

In [ ]:
# CELL: gradio
PASS_LABELS = {f"{t:%Y-%m-%d %H:%M} UTC · {p} · {o}": i for i, (t, p, o) in
               enumerate(zip(PASSES.time, PASSES.platform, PASSES.orbit))
               if all(PASSES[f"cov_{zk}"].iloc[i] for zk in ZONE_ORDER)}

def pass_figure(label):
    i = PASS_LABELS[label]
    fig, axes = plt.subplots(2, 1, figsize=(10, 8.5), gridspec_kw=dict(height_ratios=[ZONES["Mundra"]["H"] / ZONES["Mundra"]["W"],
                                                                                  ZONES["Kandla"]["H"] / ZONES["Kandla"]["W"]]))
    txt = []
    for ax, zk in zip(axes, ZONE_ORDER):
        n = draw_zone(ax, zk, i)
        ax.set_title(f"{ZONES[zk]['name']}: {n} ships", loc="left", fontsize=13)
        txt.append(f"{zk}: {n}")
    B.source_note(fig, f"{label} · {DATA_CREDIT}" + (" · SMOKE TEST" if SMOKE_TEST else ""))
    fig.tight_layout()
    return fig, " · ".join(txt)

try:
    import gradio as gr
    with gr.Blocks(title=f"23N Field Test #{FT_NUM}: ships at anchor") as demo:
        gr.Markdown(f"### 23N Field Test #{FT_NUM}: ships at anchor off Mundra and Kandla, from Sentinel-1 radar\n"
                    "Independent study on public data. Counts are detector estimates.")
        dd_ = gr.Dropdown(list(PASS_LABELS), value=list(PASS_LABELS)[-1] if PASS_LABELS else None, label="Radar pass")
        out_plot, out_txt = gr.Plot(), gr.Textbox(label="Ships detected")
        dd_.change(pass_figure, dd_, [out_plot, out_txt])
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", type(e).__name__, e)

## 6 · Video (1080×1080, 30 fps, 58 s, H.264) and cover (1080×1350)
Every image, circle and number in the video comes from the run above. The pass shown is labelled for what it is:
the pass with the most ships in the test year.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30

def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def build_video_ctx():
    both = P[(P.year == TEST_YEAR) & P[[f"cov_{zk}" for zk in ZONE_ORDER]].all(axis=1)]
    if not len(both):
        both = P[P[[f"cov_{zk}" for zk in ZONE_ORDER]].all(axis=1)]
    tot = both[ZONE_ORDER].sum(axis=1)
    i = int(tot.idxmax())
    return dict(i=i, time=PASSES.time.iloc[i], n={zk: int(P.loc[i, zk]) for zk in ZONE_ORDER})

def _title(fig, title, sub):
    fig.text(0.055, 0.85, title, fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
    fig.text(0.055, 0.785, sub.upper(), fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")

def _zone_axes(fig, zk):
    z = ZONES[zk]; aspect = z["H"] / z["W"]
    w = 0.89; h = min(0.52, w * aspect * 1.0)
    return fig.add_axes([0.055, 0.20 + (0.52 - h) / 2, w, h])

def draw_evidence(fig, t, c):
    # t runs 0..35 inside the evidence section (video seconds 5..40)
    B.header(fig, FT_NUM)
    if t < 18:                                                     # the busiest pass, zone by zone
        zk = "Mundra" if t < 9 else "Kandla"; t0 = 0 if t < 9 else 9
        n = c["n"][zk]; k = int(round(n * _ease(_prog(t, t0 + 1.2, t0 + 6))))
        _title(fig, "Mundra's anchorages" if zk == "Mundra" else "Off Kandla and Tuna",
               f"Sentinel-1 VH radar · {c['time']:%d %b %Y %H:%M} UTC · busiest pass of {TEST_YEAR}")
        ax = _zone_axes(fig, zk)
        draw_zone(ax, zk, c["i"], n_show=k)
        fig.text(0.945, 0.785, f"{k} SHIPS", fontproperties=B.FP.get("mono_bold"), fontsize=18, color=B.MOSS,
                 ha="right", va="top")
        B.caption(fig, "Radar sees through cloud, day and night. Pale is calm sea; ships are the dark points"
                  if zk == "Mundra" else "Land and fixed structures are masked using the median of every pass")
    elif t < 30:                                                   # the weekly index
        _title(fig, "Every week, two years", "mean ships per pass · Mundra + Kandla · shaded: monsoon")
        ax = fig.add_axes([0.11, 0.22, 0.84, 0.50])
        span = pd.Timestamp(DATA_END) - pd.Timestamp(DATA_START)
        draw_weekly(ax, upto=pd.Timestamp(DATA_START) + span * _ease(_prog(t, 18.3, 26)), big=True)
        B.caption(fig, f"{int(P[ZONE_ORDER].notna().all(axis=1).sum())} passes where the satellite saw both zones")
    else:                                                          # monsoon vs the rest
        _title(fig, "Monsoon vs the rest of the year", "mean ships per pass · both zones")
        ax = fig.add_axes([0.13, 0.22, 0.82, 0.50])
        g = _ease(_prog(t, 30.1, 32))
        x = np.arange(2); wdt = 0.36
        ax.bar(x - wdt / 2, [REST_R * g, REST_T * g], wdt, color=B.AMBER, label="Oct–May")
        ax.bar(x + wdt / 2, [MON_R * g, MON_T * g], wdt, color=B.MOSS, label="Jun–Sep monsoon")
        ax.set_xticks(x, [str(ROBUST_YEAR), str(TEST_YEAR)], fontsize=15); ax.tick_params(axis="y", labelsize=13)
        ax.set_ylim(0, max(REST_R, REST_T, MON_R, MON_T, 1) * 1.3); ax.set_ylabel("ships per pass", fontsize=14)
        ax.legend(loc="upper left", fontsize=14, ncol=2)
        B.caption(fig, f"{TEST_YEAR}: {fmt_n(MON_T)} ships per pass in the monsoon, {fmt_n(REST_T)} the rest of the year")

def render_video(c, path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "How many ships wait at anchor off Mundra and Kandla?",
                f"Sentinel-1 radar · {len(PASSES)} passes · {DATA_START[:4]}-{DATA_END[:4]}"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        if t < 18:   key = ("z", min(round(t * 10) / 10, 8.6 if t < 9 else 17.6))
        elif t < 30: key = ("w", min(round(t * 10) / 10, 26.2))
        else:        key = ("b", min(round(t * 10) / 10, 32.2))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t, c); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, BIG, BIG_LINE,
                               f"CA-CFAR ship detector on Sentinel-1 VH, {len(PASSES)} passes {DATA_START[:4]}-{DATA_END[:4]}, "
                               "Mundra's published anchorages + a Kandla-Tuna box. Estimate.")
    v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    print("video frames:", v.frames, f"({v.frames / FPS:.1f}s)")

def render_cover(c, path=COVER_PATH):
    def panel(ax):
        ax.set_axis_on(); draw_weekly(ax, big=True)
        pos = ax.get_position()
        ax.set_position([pos.x0 + 0.07, pos.y0 + 0.03, pos.width - 0.07, pos.height - 0.03])
    B.cover(path, FT_NUM, "Ships at anchor, seen by radar", f"Mundra + Kandla · Sentinel-1 · {DATA_START[:4]}-{DATA_END[:4]}",
            BIG, f"at anchor on an average radar pass in {TEST_YEAR}; monsoon {CHG_T:+.0%} (estimate)",
            image=panel, data_credit="Copernicus Sentinel data")

In [ ]:
# CELL: video_render
VCTX = build_video_ctx()
render_video(VCTX)
render_cover(VCTX)
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 7 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """How many ships wait at anchor off Mundra and Kandla? We counted them from radar satellites that see through cloud and the monsoon.

23N Field Test #07: {headline}

In 2024-25 the average pre-berthing detention at Deendayal Port (Kandla) was {pbdt_kandla} hours, against {pbdt_major} hours across India's major ports (Ministry of Ports, Shipping and Waterways).

How we tested it:
• {n_passes} Sentinel-1 radar passes, {period}, over Mundra's five published anchorage areas and a Kandla–Tuna study box
• A classic adaptive-threshold (CFAR) ship detector: no training data, runs on a free Colab CPU
• Weekly index = mean ships per pass; monsoon = Jun–Sep

Result: {mean_t} ships per pass in {test_year} ({mean_r} in {robust_year}). Monsoon {mon_t} vs {rest_t} the rest of the year. Busiest week: {peak_week}.

Limits: one image can't tell anchored from slow-moving ships; small craft are missed; no AIS to check against. An estimate.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """How many ships wait at anchor off Mundra and Kandla? Sentinel-1 radar, {n_passes} passes: {mean_t} ships per pass in {test_year}; monsoon {chg_t} vs the rest of the year. 23N Field Test #07. Notebook below.

---
Thread (optional):
1/ Data: Copernicus Sentinel-1 VH radar via Microsoft Planetary Computer, no login. Mundra's five published anchorage areas plus a Kandla–Tuna box we drew.
2/ Method: land masked from the median of all passes; a CFAR detector finds bright clusters on dark sea. {robust_year}: {mean_r} ships per pass.
3/ Limits: can't tell anchored from slow-moving ships; no AIS check; small craft missed. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
POST_VALUES = dict(headline=HEADLINE, n_passes=len(PASSES), period=f"{DATA_START[:4]}–{DATA_END[:4]}",
                   pbdt_kandla=f"{PBDT_KANDLA_H:.1f}", pbdt_major=f"{PBDT_MAJOR_H:.1f}",
                   mean_t=fmt_n(MEAN_T), mean_r=fmt_n(MEAN_R), test_year=TEST_YEAR, robust_year=ROBUST_YEAR,
                   mon_t=fmt_n(MON_T), rest_t=fmt_n(REST_T), chg_t=f"{CHG_T:+.0%}",
                   peak_week=(f"week of {PEAK_WEEK:%d %b %Y}, {PEAK_VAL:.0f} ships per pass" if PEAK_WEEK == PEAK_WEEK else "n/a"),
                   signature=B.post_signature(FT_NUM) + " #Shipping #Ports #EarthObservation")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 8 · Limitations (read before quoting the number)

- **"Present", not "waiting".** One radar image can't tell a ship at anchor from one moving slowly through the zone.
  In published anchorage areas most ships are at anchor; in the Kandla–Tuna box some will be under way. The count
  is ships present at the moment of the pass.
- **A snapshot every few days, not a census.** Sentinel-1 passes over the Gulf at fixed times (about 01:00 and
  13:00 UTC), every few days. The weekly index is a mean of 1–4 snapshots, not a continuous count. The number of
  satellites changed (Sentinel-1C from 2025, 1D from 2026), which changes how many snapshots a week has but not
  what each one counts; the notebook checks 1A against 1C.
- **No ground truth.** AIS ship-position data would confirm each detection, but it isn't open. Detections are
  checked only for plausibility (size, location, stability across thresholds and orbits).
- **Detector limits.** Small craft (under about 60 m) are often missed; two ships moored side by side count as one;
  strong wind in the monsoon makes the sea brighter and can hide weak targets or add false ones (hence the stricter
  threshold variant); bright targets on land can leave faint "ghost" echoes over water.
- **Zones are choices.** Mundra's areas come from the port's own booklet plus a 500 m buffer. The Kandla–Tuna box
  is ours; we found no published coordinates for Kandla's anchorages. Ships told to wait outside the Gulf of Kutch
  are not in either zone.
- **Not a statement about any port, operator or authority.** Mundra and Kandla are used only as locations.
- **Licences.** Copernicus Sentinel data terms: free, full and open; attribution "Contains modified Copernicus
  Sentinel data 2024–2025". Access via Microsoft Planetary Computer (no account). Code in this repo: see LICENSE.